In [149]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

X_train = np.loadtxt('X_train.csv', delimiter=',')
y_train = np.loadtxt('y_train.csv')
X_test = np.loadtxt('X_test.csv', delimiter=',')
y_test = np.loadtxt('y_test.csv')

1.1 Class frequencies.

The ith row in X_train.csv are the features of the ith training pattern. The
class label of the ith pattern is given in the ith row of y_train.csv.

In [75]:
# Count the occurrences of each class
unique, counts = np.unique(y_train, return_counts=True)

# Calculate the frequency of each class
class_frequencies = counts / len(y_train)

# Display results
for label, freq in zip(unique, class_frequencies):
    print(f"Class {int(label)}: {freq:.4f}")

Class 0: 0.5209
Class 1: 0.0955
Class 2: 0.2527
Class 3: 0.0469
Class 4: 0.0839


1.2 Classification

The task is to evaluate several multi-class classifiers on the data. Build the models
using the training data only. The test data must only be used for final evaluation.

1.2.1 Logistic regression

Apply multi-nominal logistic regression. If you use regularization, describe the
type of regularization you used. Report training and test loss (in terms of 0-1
loss).

In [93]:
# Train logistic regression
logreg = LogisticRegression(penalty='l2', fit_intercept=True, multi_class='multinomial', solver='lbfgs', max_iter=1000)
logreg.fit(X_train, y_train);

# Make predictions on training and test data
y_train_pred = logreg.predict(X_train)
y_test_pred = logreg.predict(X_test)

# Calculate training and test accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Calculate training and test loss (0-1 loss)
train_loss = 1 - train_accuracy
test_loss = 1 - test_accuracy

# Print results
print(f'Training Loss: {train_loss:.4f}')
print(f'Test Loss: {test_loss:.4f}')

Training Loss: 0.1496
Test Loss: 0.0993


1.2.2 Random forest

Apply random forests with 50, 100, and 200 trees. Report training and test loss
(in terms of 0-1 loss) as well as out-of-bag error.

In [151]:
n_trees = [50,100,200]
ret_val = {}

# Apply random forest
for n in n_trees:
    rf = RandomForestClassifier(n_estimators=n, max_features='sqrt', oob_score=True, max_samples=1.0)
    rf.fit(X_train, y_train)

    rf_y_train_pred = rf.predict(X_train)
    rf_y_test_pred = rf.predict(X_test)

    # Calculate accuracy
    train_accuracy = accuracy_score(y_train, rf_y_train_pred)
    test_accuracy = accuracy_score(y_test, rf_y_test_pred)

    ret_val[n] = {
        'train_loss': 1 - train_accuracy,
        'test_loss': 1 - test_accuracy,
        'oob_error': 1 - rf.oob_score_
    }

# Print results
for trees, metrics in ret_val.items():
    print(f'Trees: {trees}')
    print(f'  Training loss: {metrics["train_loss"]:.4f}')
    print(f'  Test loss: {metrics["test_loss"]:.4f}')
    print(f'  Out-of-Bag error: {metrics["oob_error"]:.4f}')


Trees: 50
  Training loss: 0.0004
  Test loss: 0.1129
  Out-of-Bag error: 0.1532
Trees: 100
  Training loss: 0.0000
  Test loss: 0.1134
  Out-of-Bag error: 0.1498
Trees: 200
  Training loss: 0.0000
  Test loss: 0.1093
  Out-of-Bag error: 0.1472


1.2.3 Nearest neighbor

Apply k-nearest-neighbor classification. Use cross-validation to determine the
number of neighbors. Report training and test loss (in terms of 0-1 loss). Describe
how you determined the number of neighbors.

In [148]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

# Find optimal k using adaptive stopping
max_k = 500
optimal_k = 1
best_score = 0.0
no_improvement_threshold = 10
no_improvement_count = 0

# Iterate through possible values of k
for k in range(1, max_k + 1):
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # 5-fold CV
    cv_scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy') 
    mean_score = np.mean(cv_scores)
    
    # If improvement was found
    if mean_score > best_score:
        best_score = mean_score
        optimal_k = k
        print(f'new optimal k={k}')
        no_improvement_count = 0
    else:
        no_improvement_count += 1

    # Stop if threshold reached
    if no_improvement_count >= no_improvement_threshold:
        break

print(f'optimal k={optimal_k}')
knn = KNeighborsClassifier(n_neighbors=optimal_k)
knn.fit(X_train, y_train)

y_train_pred = knn.predict(X_train)
y_test_pred = knn.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_loss = 1 - train_accuracy
test_loss = 1 - test_accuracy

print(f'Training loss: {train_loss:.4f}')
print(f'Test loss: {test_loss:.4f}')

new optimal k=1
new optimal k=2
new optimal k=3
new optimal k=4
new optimal k=5
new optimal k=6
new optimal k=7
new optimal k=8
new optimal k=9
new optimal k=10
new optimal k=12
new optimal k=14
new optimal k=15
new optimal k=17
new optimal k=18
new optimal k=19
new optimal k=21
new optimal k=22
new optimal k=24
new optimal k=26
new optimal k=30
new optimal k=31
new optimal k=32
new optimal k=34
new optimal k=38
new optimal k=40
new optimal k=42
new optimal k=46
new optimal k=51
new optimal k=52
optimal k=52
Training loss: 0.1448
Test loss): 0.0983


optimal k=52
